In [2]:
#%pip install -U jupyter ipywidgets

In [3]:
#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [4]:
#%pip install --no-deps --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


In [5]:
# Cell 1: Install all required packages
%pip install transformers datasets evaluate jiwer accelerate peft -q

print("✅ All packages installed!")
print("   - transformers: Whisper model")
print("   - datasets: Data loading")
print("   - evaluate: WER metric")
print("   - peft: LoRA fine-tuning")

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed!
   - transformers: Whisper model
   - datasets: Data loading
   - evaluate: WER metric
   - peft: LoRA fine-tuning



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Cell 2: Import all required libraries
import os
import json
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import Dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate
from pathlib import Path
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✅ All libraries imported successfully!
PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.59 GB


In [7]:
# Cell 3: Configuration parameters
MODEL_NAME = "openai/whisper-small"  # tiny, base, small, medium, large
LANGUAGE = "en"
TASK = "transcribe"
OUTPUT_DIR = "./whisper-torgo-lora"
DATA_DIR = "torgo_prepared_FULL"  # Change from "torgo_local"

# LoRA-specific settings
LORA_R = 32  # LoRA rank (8, 16, 32, 64)
LORA_ALPHA = 64  # LoRA alpha (typically 2x rank)
LORA_DROPOUT = 0.05

print("="*60)
print("📋 CONFIGURATION")
print("="*60)
print(f"Model: {MODEL_NAME}")
print(f"Language: {LANGUAGE}")
print(f"Task: {TASK}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"\n🔧 LoRA Settings:")
print(f"   Rank (r): {LORA_R}")
print(f"   Alpha: {LORA_ALPHA}")
print(f"   Dropout: {LORA_DROPOUT}")
print("="*60)

📋 CONFIGURATION
Model: openai/whisper-small
Language: en
Task: transcribe
Output directory: ./whisper-torgo-lora
Data directory: torgo_prepared_FULL

🔧 LoRA Settings:
   Rank (r): 32
   Alpha: 64
   Dropout: 0.05


In [8]:
# NEW CELL: Fix paths to point to train_wavs and val_wavs
import json
from pathlib import Path

def fix_jsonl_paths(input_file, output_file, wavs_folder):
    """Update paths to point to train_wavs or val_wavs folders"""
    fixed_data = []
    
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            
            # Extract just the filename
            filename = Path(item['audio_filepath']).name
            
            # Create new path pointing to wavs folder
            item['audio_filepath'] = str(Path(DATA_DIR) / wavs_folder / filename)
            
            fixed_data.append(item)
    
    # Write fixed data
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in fixed_data:
            f.write(json.dumps(item) + '\n')
    
    return len(fixed_data)

# Fix both files
train_count = fix_jsonl_paths(
    Path(DATA_DIR) / "train.jsonl",
    Path(DATA_DIR) / "train_fixed.jsonl",
    "train_wavs"
)

val_count = fix_jsonl_paths(
    Path(DATA_DIR) / "val.jsonl",
    Path(DATA_DIR) / "val_fixed.jsonl",
    "val_wavs"
)

print(f"✅ Fixed {train_count} training paths → train_wavs/")
print(f"✅ Fixed {val_count} validation paths → val_wavs/")

# Verify
with open(Path(DATA_DIR) / "train_fixed.jsonl", 'r') as f:
    sample = json.loads(f.readline())
    print(f"\n📋 Sample path: {sample['audio_filepath']}")
    print(f"   Exists: {Path(sample['audio_filepath']).exists()}")

✅ Fixed 14896 training paths → train_wavs/
✅ Fixed 1656 validation paths → val_wavs/

📋 Sample path: torgo_prepared_FULL\train_wavs\sample_10018.wav
   Exists: True


In [9]:
# Cell 4: Function to load JSONL dataset
def load_jsonl_dataset(jsonl_path):
    """
    Load dataset from JSONL file.
    Each line: {"path": "audio.wav", "transcript": "text", ...}
    """
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            data.append(item)
    
    dataset = Dataset.from_list(data)
    return dataset

# Test loading
try:
    test_data = load_jsonl_dataset(Path(DATA_DIR) / "train.jsonl")
    print(f"✅ Data loading function works!")
    print(f"   Found {len(test_data)} training samples")
    print(f"\n📄 Sample data:")
    print(f"   {test_data[0]}")
except Exception as e:
    print(f"⚠️ Error loading data: {e}")
    print(f"   Make sure {DATA_DIR}/train.jsonl exists")

✅ Data loading function works!
   Found 14896 training samples

📄 Sample data:
   {'audio_filepath': '/content/drive/MyDrive/torgo_prepared_FULL/train_wavs/sample_10018.wav', 'text': 'sport', 'duration': 0.672}


In [10]:
# Cell 5: Function to prepare audio for Whisper
def prepare_dataset(batch, processor):
    """
    Convert raw audio to mel spectrogram and tokenize transcript.
    """
    # Load audio
    audio = batch["audio"]
    
    # Convert audio to mel spectrogram (Whisper's input format)
    batch["input_features"] = processor.feature_extractor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    # Convert transcript to token IDs
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    
    return batch

print("✅ Data preparation function defined")
print("   Converts: Audio waveform → Mel spectrogram")
print("   Converts: Text → Token IDs")

✅ Data preparation function defined
   Converts: Audio waveform → Mel spectrogram
   Converts: Text → Token IDs


In [11]:
# Cell 6: Data collator (float16 for GPU)
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class WhisperDataCollatorFixed:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = torch.stack([
            torch.tensor(f["input_features"], dtype=torch.float16)  # Match model dtype
            for f in features
        ])
        
        label_lengths = [len(f["labels"]) for f in features]
        max_label_length = max(label_lengths)
        
        labels = []
        for f in features:
            label = f["labels"]
            padded = label + [-100] * (max_label_length - len(label))
            labels.append(padded)
        
        labels = torch.tensor(labels, dtype=torch.long)
        
        return {
            "input_features": input_features,
            "labels": labels
        }

print("✅ Collator ready")

✅ Collator ready


In [12]:
# Cell 7: WER (Word Error Rate) metric
def compute_metrics(pred, tokenizer, metric):
    """
    Calculate Word Error Rate (WER).
    Lower is better: 0% = perfect, 100% = completely wrong
    """
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad token
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode predictions and references to text
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Calculate WER
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

print("✅ WER metric function defined")
print("   Measures transcription accuracy")
print("   Example: WER=10% means 90% words correct")

✅ WER metric function defined
   Measures transcription accuracy
   Example: WER=10% means 90% words correct


In [13]:
# Cell: Find the downloaded model location
from huggingface_hub import snapshot_download
from pathlib import Path

# This will return the path where it was downloaded
model_path = snapshot_download(
    repo_id="openai/whisper-small",
    allow_patterns=[
        "*.json", "*.bin", "*.safetensors", "*.txt", "*.md"
    ],
)

print("="*60)
print("✅ MODEL LOCATION FOUND")
print("="*60)
print(f"Path: {model_path}")
print("="*60)

# Save this path for later use
LOCAL_MODEL_PATH = model_path
print(f"\n💡 Your LOCAL_MODEL_PATH is:")
print(f'LOCAL_MODEL_PATH = "{LOCAL_MODEL_PATH}"')


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

✅ MODEL LOCATION FOUND
Path: C:\Users\ASUS\.cache\huggingface\hub\models--openai--whisper-small\snapshots\973afd24965f72e36ca33b3055d56a652f456b4d

💡 Your LOCAL_MODEL_PATH is:
LOCAL_MODEL_PATH = "C:\Users\ASUS\.cache\huggingface\hub\models--openai--whisper-small\snapshots\973afd24965f72e36ca33b3055d56a652f456b4d"


In [14]:
# Cell 8: Load Whisper processor and tokenizer (Memory-Safe)
import gc
import torch

logger.info("Loading Whisper processor and tokenizer...")

# Clear any existing memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_PATH = "openai/whisper-small"

try:
    # Feature extractor: audio → mel spectrogram
    feature_extractor = WhisperFeatureExtractor.from_pretrained(
        MODEL_PATH,
        local_files_only=True
    )
    print("✅ Feature extractor loaded")
    
    # Tokenizer: text ↔ token IDs
    tokenizer = WhisperTokenizer.from_pretrained(
        MODEL_PATH,
        language=LANGUAGE, 
        task=TASK,
        local_files_only=True
    )
    print("✅ Tokenizer loaded")
    
    # Processor: combines both
    processor = WhisperProcessor.from_pretrained(
        MODEL_PATH,
        language=LANGUAGE, 
        task=TASK,
        local_files_only=True
    )
    print("✅ Processor loaded from cache!")
    
except Exception as e:
    print(f"❌ Error loading processor: {e}")
    print("\nTrying alternative method...")
    
    # Alternative: Load from explicit cache path
    from huggingface_hub import snapshot_download
    cache_path = snapshot_download(
        repo_id=MODEL_PATH,
        allow_patterns=["*.json", "merges.txt", "vocab.json", "tokenizer.json"]
    )
    
    feature_extractor = WhisperFeatureExtractor.from_pretrained(cache_path)
    tokenizer = WhisperTokenizer.from_pretrained(cache_path, language=LANGUAGE, task=TASK)
    processor = WhisperProcessor.from_pretrained(cache_path, language=LANGUAGE, task=TASK)
    
    print("✅ Processor loaded via alternative method!")

2025-10-26 05:52:30,812 - Loading Whisper processor and tokenizer...


✅ Feature extractor loaded
✅ Tokenizer loaded
✅ Processor loaded from cache!


In [15]:
# Cell: Check if you have enough RAM
import psutil

ram = psutil.virtual_memory()
available_gb = ram.available / 1e9

print(f"Available RAM: {available_gb:.2f} GB")

# whisper-small needs ~4GB RAM to load
if available_gb < 4:
    print("\n❌ NOT ENOUGH RAM!")
    print(f"   Need: 4+ GB")
    print(f"   Available: {available_gb:.1f} GB")
    print("\n📋 Options:")
    print("   1. Close other applications")
    print("   2. Restart computer")
    print("   3. Use whisper-tiny instead")
elif available_gb < 6:
    print("\n⚠️ TIGHT ON RAM")
    print("   Might work, but could crash during training")
    print("   Recommendation: Close browser/apps")
else:
    print("\n✅ Sufficient RAM available")
    print("   Should be able to load whisper-small")

Available RAM: 7.20 GB

✅ Sufficient RAM available
   Should be able to load whisper-small


In [16]:
# Cell 9: Load model with LoRA (CLEAN RESTART)
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model
import torch
import gc

# Clear everything
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading fresh model...")

# Load model directly on GPU with float16
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small",
    local_files_only=True
).to("cuda").to(torch.float16)

# Configure
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

# Apply LoRA
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)

# CRITICAL: Print LoRA modules to verify
model.print_trainable_parameters()

# Set to training mode
model.train()

# Verify parameters
print("\n🔍 Checking first LoRA parameter:")
for name, param in model.named_parameters():
    if 'lora' in name and param.requires_grad:
        print(f"  {name}")
        print(f"  requires_grad: {param.requires_grad}")
        print(f"  is_leaf: {param.is_leaf}")
        print(f"  dtype: {param.dtype}")
        print(f"  device: {param.device}")
        break

print("\n✅ Model loaded on GPU with LoRA")

Loading fresh model...
trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429

🔍 Checking first LoRA parameter:
  base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight
  requires_grad: True
  is_leaf: True
  dtype: torch.float32
  device: cuda:0

✅ Model loaded on GPU with LoRA


In [17]:
# Cell 10: Load and prepare datasets
logger.info("Loading train and validation datasets...")

# Load JSONL files
train_dataset = load_jsonl_dataset(Path(DATA_DIR) / "train_fixed.jsonl")
val_dataset = load_jsonl_dataset(Path(DATA_DIR) / "val_fixed.jsonl")

print(f"\n📊 Dataset Statistics:")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Validation samples: {len(val_dataset)}")

# Configure audio loading (16kHz for Whisper)
train_dataset = train_dataset.cast_column("path", Audio(sampling_rate=16000))
val_dataset = val_dataset.cast_column("path", Audio(sampling_rate=16000))

# Rename 'path' to 'audio' (expected by preprocessing)
train_dataset = train_dataset.rename_column("path", "audio")
val_dataset = val_dataset.rename_column("path", "audio")

print("✅ Datasets loaded and configured for 16kHz audio")

2025-10-26 05:52:32,661 - Loading train and validation datasets...



📊 Dataset Statistics:
   Training samples: 14896
   Validation samples: 1656
✅ Datasets loaded and configured for 16kHz audio


In [18]:
# Cell: Install all audio processing libraries
%pip install librosa soundfile audioread

print("✅ Installed!")

Note: you may need to restart the kernel to use updated packages.
✅ Installed!



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
print("Train dataset columns:", train_dataset.column_names[:10])
print("Val dataset columns:", val_dataset.column_names[:10])

print("\nSample train data row:")
print(train_dataset[0])


Train dataset columns: ['audio_filepath', 'text', 'duration', 'audio']
Val dataset columns: ['audio_filepath', 'text', 'duration', 'audio']

Sample train data row:
{'audio_filepath': 'torgo_prepared_FULL\\train_wavs\\sample_10018.wav', 'text': 'sport', 'duration': 0.672, 'audio': None}


In [22]:
# Cell 10: Load and preprocess with disk caching
import json
from pathlib import Path
from datasets import Dataset

logger.info("Loading datasets...")

def load_jsonl_simple(jsonl_path):
    """Load JSONL without any Audio column casting"""
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            data.append(item)
    return data

# Load raw data
train_data = load_jsonl_simple(Path(DATA_DIR) / "train_fixed.jsonl")
val_data = load_jsonl_simple(Path(DATA_DIR) / "val_fixed.jsonl")

print(f"✅ Loaded {len(train_data)} training samples")
print(f"✅ Loaded {len(val_data)} validation samples")
print(f"\n📋 Sample data: {train_data[0]}")

def preprocess_batch(examples):
    """Process a batch of examples with raw file paths"""
    # Import inside function for multiprocessing compatibility
    import librosa
    from transformers import WhisperProcessor
    
    input_features = []
    labels = []
    
    for path, text in zip(examples["audio_filepath"], examples["text"]):
        try:
            # Load and resample audio
            audio_array, _ = librosa.load(path, sr=16000)
            
            # Extract mel spectrograms for Whisper
            features = processor.feature_extractor(
                audio_array,
                sampling_rate=16000
            ).input_features[0]
            input_features.append(features)
            
            # Tokenize the transcript
            label = processor.tokenizer(text).input_ids
            labels.append(label)
        
        except Exception as e:
            print(f"Error loading {path}: {e}")
            raise
    
    return {
        "input_features": input_features,
        "labels": labels
    }

# Create Dataset objects
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print("\n🔄 Preprocessing datasets with disk caching...")

# Process with disk caching - MUST use num_proc=None (single process)
train_dataset = train_dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=4,
    remove_columns=["audio_filepath", "text", "duration"],
    desc="Processing training data",
    cache_file_name=str(Path(DATA_DIR) / "train_processed_cache.arrow"),
    writer_batch_size=50,
    num_proc=None,  # Single process to avoid serialization issues
)

val_dataset = val_dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=4,
    remove_columns=["audio_filepath", "text", "duration"],
    desc="Processing validation data",
    cache_file_name=str(Path(DATA_DIR) / "val_processed_cache.arrow"),
    writer_batch_size=50,
    num_proc=None,  # Single process
)

print("✅ Preprocessing complete!")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

2025-10-26 06:02:13,752 - Loading datasets...


✅ Loaded 14896 training samples
✅ Loaded 1656 validation samples

📋 Sample data: {'audio_filepath': 'torgo_prepared_FULL\\train_wavs\\sample_10018.wav', 'text': 'sport', 'duration': 0.672}

🔄 Preprocessing datasets with disk caching...


Processing training data:   0%|          | 0/14896 [00:00<?, ? examples/s]

Processing validation data:   0%|          | 0/1656 [00:00<?, ? examples/s]

✅ Preprocessing complete!
Training samples: 14896
Validation samples: 1656


In [23]:
# Cell 12: Reinitialize collator
data_collator = WhisperDataCollatorFixed(processor=processor)
metric = evaluate.load("wer")
print("✅ Collator reinitialized")

✅ Collator reinitialized


In [24]:
# Cell 13: Training args with remove_unused_columns=True
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    
    learning_rate=1e-3,
    warmup_steps=1,
    weight_decay=0.01,
    
    max_steps=5,
    
    fp16=True,
    dataloader_num_workers=0,
    
    eval_strategy="no",
    save_strategy="steps",
    save_steps=5,
    logging_steps=1,
    
    load_best_model_at_end=False,
    save_total_limit=1,
    
    predict_with_generate=False,
    
    report_to=[],
    push_to_hub=False,
    
    remove_unused_columns=True,  # ✅ Changed to True
    label_names=["labels"],
)

print("=" * 60)
print("✅ TRAINING CONFIGURATION (FIXED)")
print("=" * 60)
print(f"remove_unused_columns: {training_args.remove_unused_columns}")
print("=" * 60)

✅ TRAINING CONFIGURATION (FIXED)
remove_unused_columns: True


In [26]:
# Cell: Install TensorBoard and test
%pip install tensorboard

# Verify installation
import tensorboard
print(f"✅ TensorBoard version: {tensorboard.__version__}")

Note: you may need to restart the kernel to use updated packages.
✅ TensorBoard version: 2.20.0



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
# Cell 14: Reinitialize Trainer
import gc
torch.cuda.empty_cache()
gc.collect()

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=lambda pred: compute_metrics(pred, tokenizer, metric),
    tokenizer=processor.feature_extractor,
)

print("✅ Trainer reinitialized with NEW collator")

✅ Trainer reinitialized with NEW collator


C:\Users\ASUS\AppData\Local\Temp\ipykernel_476\3242630339.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [28]:
# Training function (FULL VERSION - no step limit)
def train_whisper_lora_custom(
    model,
    train_dataset,
    val_dataset,
    processor,
    output_dir="./whisper-torgo-lora",
    num_epochs=5,
    batch_size=8,
    learning_rate=5e-4
):
    from torch.utils.data import DataLoader
    from torch.optim import AdamW
    import torch
    
    device = "cuda"
    model = model.to(device)
    model.train()
    
    print(f"Model device: {next(model.parameters()).device}")
    print(f"Model dtype: {next(model.parameters()).dtype}")
    
    data_collator = WhisperDataCollatorFixed(processor=processor)
    
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
        shuffle=True
    )
    
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
        shuffle=False
    )
    
    # Get ONLY LoRA parameters
    lora_params = [p for p in model.parameters() if p.requires_grad]
    print(f"Total trainable params: {len(lora_params)}")
    
    optimizer = AdamW(lora_params, lr=learning_rate)
    
    print("\n" + "="*60)
    print("🚀 STARTING TRAINING")
    print("="*60 + "\n")
    
    for epoch in range(num_epochs):
        print(f"\n📍 Epoch {epoch+1}/{num_epochs}")
        
        # Training
        model.train()
        total_loss = 0
        num_steps = 0
        
        for step, batch in enumerate(train_dataloader):
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)
            
            optimizer.zero_grad()
            
            outputs = model(
                input_features=input_features,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            num_steps += 1
            
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(lora_params, max_norm=1.0)
            
            optimizer.step()
            
            if (step + 1) % 10 == 0:
                print(f"  Step {step+1}/{len(train_dataloader)}: Loss = {loss.item():.4f}")
        
        avg_train_loss = total_loss / num_steps
        print(f"  ✅ Avg Training Loss: {avg_train_loss:.4f}")
        
        # Validation
        model.eval()
        val_loss = 0
        val_steps = 0
        
        with torch.no_grad():
            for batch in val_dataloader:
                input_features = batch["input_features"].to(device)
                labels = batch["labels"].to(device)
                
                outputs = model(
                    input_features=input_features,
                    labels=labels
                )
                
                val_loss += outputs.loss.item()
                val_steps += 1
        
        avg_val_loss = val_loss / val_steps
        print(f"  ✅ Avg Validation Loss: {avg_val_loss:.4f}")
    
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETE!")
    print("="*60)
    
    # Save
    model.save_pretrained(output_dir)
    processor.save_pretrained(output_dir)
    print(f"\n💾 Model saved to {output_dir}")
    
    return model

print("✅ Full training function ready")

✅ Full training function ready


In [29]:
# Diagnostic: Check what's actually happening
import torch
import psutil

print("=" * 60)
print("SYSTEM STATUS")
print("=" * 60)

# RAM
ram = psutil.virtual_memory()
print(f"RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB ({ram.percent}%)")

# GPU
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1e9:.2f} GB allocated")
    print(f"GPU Memory: {torch.cuda.memory_reserved(0)/1e9:.2f} GB reserved")
else:
    print("⚠️ No GPU detected - running on CPU (VERY SLOW)")

# Check if model is on GPU
print(f"\nModel device: {next(model.parameters()).device}")
print(f"Model dtype: {next(model.parameters()).dtype}")

# Check batch
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))
print(f"\nBatch size: {batch['input_features'].shape[0]}")
print(f"Batch device: {batch['input_features'].device}")

print("=" * 60)

SYSTEM STATUS
RAM: 11.4/16.8 GB (67.9%)
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 0.54 GB allocated
GPU Memory: 1.00 GB reserved

Model device: cuda:0
Model dtype: torch.float16

Batch size: 8
Batch device: cuda:0


In [30]:
# Debug: Test the collator directly
print("Testing collator...")
test_features = [train_dataset[0], train_dataset[1]]
batch = data_collator(test_features)

print("\n✅ Batch keys:", list(batch.keys()))
print("\nBatch contents:")
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
    else:
        print(f"  {key}: {type(value)}")

Testing collator...

✅ Batch keys: ['input_features', 'labels']

Batch contents:
  input_features: shape torch.Size([2, 80, 3000]), dtype torch.float16
  labels: shape torch.Size([2, 5]), dtype torch.int64


In [31]:
# Debug: Check what columns are in the preprocessed dataset
print("Dataset columns:", train_dataset.column_names)
print("Dataset features:", train_dataset.features)
print("\nFirst sample keys:", train_dataset[0].keys())

Dataset columns: ['input_features', 'labels']
Dataset features: {'input_features': List(List(Value('float32'))), 'labels': List(Value('int64'))}

First sample keys: dict_keys(['input_features', 'labels'])


In [32]:
# Debug cell - Check gradient status
print("🔍 Debugging gradient status:\n")

total_params = 0
trainable_params = 0
params_with_grad = 0

for name, param in model.named_parameters():
    total_params += 1
    if param.requires_grad:
        trainable_params += 1
        if param.grad_fn is not None or param.is_leaf:
            params_with_grad += 1
    
    # Print first 10 parameters
    if total_params <= 10:
        print(f"{name}: requires_grad={param.requires_grad}, is_leaf={param.is_leaf}")

print(f"\n📊 Summary:")
print(f"Total parameters: {total_params}")
print(f"Trainable (requires_grad=True): {trainable_params}")
print(f"Ready for backprop: {params_with_grad}")

if trainable_params == 0:
    print("\n❌ NO TRAINABLE PARAMETERS! Need to fix model loading.")
else:
    print("\n✅ Model has trainable parameters")

🔍 Debugging gradient status:

base_model.model.model.encoder.conv1.weight: requires_grad=False, is_leaf=True
base_model.model.model.encoder.conv1.bias: requires_grad=False, is_leaf=True
base_model.model.model.encoder.conv2.weight: requires_grad=False, is_leaf=True
base_model.model.model.encoder.conv2.bias: requires_grad=False, is_leaf=True
base_model.model.model.encoder.embed_positions.weight: requires_grad=False, is_leaf=True
base_model.model.model.encoder.layers.0.self_attn.k_proj.weight: requires_grad=False, is_leaf=True
base_model.model.model.encoder.layers.0.self_attn.v_proj.base_layer.weight: requires_grad=False, is_leaf=True
base_model.model.model.encoder.layers.0.self_attn.v_proj.base_layer.bias: requires_grad=False, is_leaf=True
base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight: requires_grad=True, is_leaf=True
base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight: requires_grad=True, is_leaf=True

📊 Summary:
Total paramete

In [33]:
# Cell 15: FULL TRAINING
model = train_whisper_lora_custom(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    processor=processor,
    output_dir=OUTPUT_DIR,
    num_epochs=5,  # Train for 5 epochs
    batch_size=8,  # Increase batch size
    learning_rate=5e-4
)

print("\n🎉 Full training complete!")

Model device: cuda:0
Model dtype: torch.float16
Total trainable params: 144

🚀 STARTING TRAINING


📍 Epoch 1/5
  Step 10/1862: Loss = 6.0820
  Step 20/1862: Loss = 4.1523
  Step 30/1862: Loss = 3.2637
  Step 40/1862: Loss = 2.3633
  Step 50/1862: Loss = 1.3604
  Step 60/1862: Loss = 0.6680
  Step 70/1862: Loss = 0.1577
  Step 80/1862: Loss = 0.8232
  Step 90/1862: Loss = 1.4482
  Step 100/1862: Loss = 3.3730
  Step 110/1862: Loss = 5.8594
  Step 120/1862: Loss = 2.8945
  Step 130/1862: Loss = 4.6094
  Step 140/1862: Loss = 3.2715
  Step 150/1862: Loss = 2.5059
  Step 160/1862: Loss = 4.6680
  Step 170/1862: Loss = 3.6934
  Step 180/1862: Loss = 1.8604
  Step 190/1862: Loss = 1.9131
  Step 200/1862: Loss = 0.6533
  Step 210/1862: Loss = 0.7036
  Step 220/1862: Loss = 0.9507
  Step 230/1862: Loss = 0.3665
  Step 240/1862: Loss = 0.3040
  Step 250/1862: Loss = 0.3682
  Step 260/1862: Loss = 0.0898
  Step 270/1862: Loss = 0.7598
  Step 280/1862: Loss = 0.5977
  Step 290/1862: Loss = 0.1318

In [34]:
# Cell 16: Save the fine-tuned LoRA model
logger.info("Saving fine-tuned model...")

# Save LoRA adapter weights (tiny files!)
model.save_pretrained(OUTPUT_DIR)

# Save processor
processor.save_pretrained(OUTPUT_DIR)

# Save tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

print("\n" + "="*60)
print("✅ MODEL SAVED")
print("="*60)
print(f"Location: {OUTPUT_DIR}")
print(f"\nSaved files:")
print(f"  - adapter_model.bin (LoRA weights, ~10-50MB)")
print(f"  - adapter_config.json (LoRA config)")
print(f"  - preprocessor_config.json")
print(f"  - tokenizer files")
print("="*60)

2025-10-26 08:34:18,849 - Saving fine-tuned model...



✅ MODEL SAVED
Location: ./whisper-torgo-lora

Saved files:
  - adapter_model.bin (LoRA weights, ~10-50MB)
  - adapter_config.json (LoRA config)
  - preprocessor_config.json
  - tokenizer files


In [35]:
# Cell 19: Check GPU memory usage
if torch.cuda.is_available():
    print("="*60)
    print("💾 GPU MEMORY SUMMARY")
    print("="*60)
    
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"Total GPU memory: {total:.2f} GB")
    print(f"Allocated: {allocated:.2f} GB ({100*allocated/total:.1f}%)")
    print(f"Reserved: {reserved:.2f} GB ({100*reserved/total:.1f}%)")
    print(f"Free: {total - reserved:.2f} GB")
    print("="*60)
    
    print("\n💡 With LoRA, you're using much less memory than full fine-tuning!")
else:
    print("⚠️ No GPU detected")

💾 GPU MEMORY SUMMARY
Total GPU memory: 8.59 GB
Allocated: 0.57 GB (6.6%)
Reserved: 5.54 GB (64.5%)
Free: 3.05 GB

💡 With LoRA, you're using much less memory than full fine-tuning!


In [36]:
# Cell 20: What to do next
print("="*60)
print("🎉 CONGRATULATIONS! YOUR MODEL IS TRAINED")
print("="*60)
print("\n📋 Next Steps:")
print("\n1. Load your model for inference:")
print("   ```python")
print("   from transformers import WhisperForConditionalGeneration, WhisperProcessor")
print("   from peft import PeftModel")
print(f"   ")
print(f"   base_model = WhisperForConditionalGeneration.from_pretrained('{MODEL_NAME}')")
print(f"   model = PeftModel.from_pretrained(base_model, '{OUTPUT_DIR}')")
print(f"   processor = WhisperProcessor.from_pretrained('{OUTPUT_DIR}')")
print("   ```")
print("\n2. Transcribe new audio:")
print("   Use the transcribe_audio() function from Chunk 18")
print("\n3. Share your model:")
print("   - Upload to HuggingFace Hub")
print("   - Share LoRA adapters (tiny files!)")
print("\n4. Further improvements:")
print("   - Train longer (increase max_steps)")
print("   - Use larger model (medium/large)")
print("   - Add more training data")
print("   - Tune LoRA hyperparameters (rank, alpha)")
print("\n5. View training logs:")
print(f"   tensorboard --logdir={OUTPUT_DIR}")
print("="*60)

🎉 CONGRATULATIONS! YOUR MODEL IS TRAINED

📋 Next Steps:

1. Load your model for inference:
   ```python
   from transformers import WhisperForConditionalGeneration, WhisperProcessor
   from peft import PeftModel
   
   base_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')
   model = PeftModel.from_pretrained(base_model, './whisper-torgo-lora')
   processor = WhisperProcessor.from_pretrained('./whisper-torgo-lora')
   ```

2. Transcribe new audio:
   Use the transcribe_audio() function from Chunk 18

3. Share your model:
   - Upload to HuggingFace Hub
   - Share LoRA adapters (tiny files!)

4. Further improvements:
   - Train longer (increase max_steps)
   - Use larger model (medium/large)
   - Add more training data
   - Tune LoRA hyperparameters (rank, alpha)

5. View training logs:
   tensorboard --logdir=./whisper-torgo-lora
